In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import uniform, randint

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

import pickle

In [ ]:
target_train = pd.read_parquet("../data/target_train.parquet")
network_train = pd.read_parquet("../data/network_train.parquet")
weather_train = pd.read_parquet("../data/weather_train.parquet")
weather_test = pd.read_parquet("../data/weather_test.parquet")
network_test = pd.read_parquet("../data/network_test.parquet")

In [ ]:
def flatten_weather_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = ["_".join([str(i) for i in col]) for col in df.columns]
    return df

def interpolate_weather(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_index()
    df = df.interpolate(method="time")
    df = df.ffill()
    return df

def aggregate_weather(df: pd.DataFrame) -> pd.DataFrame:
    ssrd_cols = [c for c in df.columns if c.endswith("ssrd")]
    tcc_cols  = [c for c in df.columns if c.endswith("tcc")]
    temp_cols = [c for c in df.columns if c.endswith("2t")]
    wind_cols = [c for c in df.columns if c.endswith("100ws")]

    flat = pd.DataFrame(index=df.index)
    flat["ssrd_mean"] = df[ssrd_cols].mean(axis=1)
    flat["ssrd_std"]  = df[ssrd_cols].std(axis=1)
    flat["tcc_mean"]  = df[tcc_cols].mean(axis=1)
    flat["tcc_std"]   = df[tcc_cols].std(axis=1)
    flat["temp_mean"] = df[temp_cols].mean(axis=1)
    flat["temp_std"]  = df[temp_cols].std(axis=1)
    flat["wind_mean"] = df[wind_cols].mean(axis=1)
    flat["wind_std"]  = df[wind_cols].std(axis=1)

    return flat.bfill().ffill()

def prepare_weather(df: pd.DataFrame) -> pd.DataFrame:
    """Flatten multi-index columns, interpolate, and aggregate to per-hour stats."""
    df = flatten_weather_columns(df)
    df = interpolate_weather(df)
    df = aggregate_weather(df)
    df = df.reset_index().rename(columns={df.index.name or "index": "ds"})
    return df

def add_lag_features_load(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["temp_lag1"]  = df["temp_mean"].shift(1)
    df["temp_lag24"] = df["temp_mean"].shift(24)

    return df

def add_rolling_features_load(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Temperature rolling (
    df["temp_roll_3h"]     = df["temp_mean"].rolling(3).mean()
    df["temp_roll_6h"]     = df["temp_mean"].rolling(6).mean()
    df["temp_roll_24h"]    = df["temp_mean"].rolling(24).mean()
    df["temp_roll_3h_std"] = df["temp_mean"].rolling(3).std()
    return df

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dt = pd.to_datetime(df["ds"])

    hour = dt.dt.hour
    dayofweek = dt.dt.dayofweek
    month = dt.dt.month

    # Cyclical encoding
    df["hour_sin"]  = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * hour / 24)
    df["dow_sin"]   = np.sin(2 * np.pi * dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * dayofweek / 7)
    df["month_sin"] = np.sin(2 * np.pi * month / 12)
    df["month_cos"] = np.cos(2 * np.pi * month / 12)

    return df
def engineer_weather_features_load(df: pd.DataFrame) -> pd.DataFrame:
    """Complete feature engineering pipeline FOR LOAD."""
    df = add_lag_features_load(df)
    df = add_rolling_features_load(df)
    df = add_time_features_load(df)
    return df.dropna()


 Preparing weather data- I will use only Temperature

In [ ]:
weather_processed = prepare_weather(weather_train)
weather_load = weather_processed[['ds', 'temp_mean', 'temp_std']].copy()

In [ ]:
weather_load.head()

,ds,temp_mean,temp_std
0,2020-01-01 00:00:00+00:00,3.114028,3.679530
1,2020-01-01 01:00:00+00:00,2.807910,3.637566
2,2020-01-01 02:00:00+00:00,2.749704,3.592852
3,2020-01-01 03:00:00+00:00,2.656238,3.711994
4,2020-01-01 04:00:00+00:00,2.599818,3.779075


In [ ]:
weather_load.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43849 entries, 0 to 43848
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype              
---  ------     --------------  -----              
 0   ds         43849 non-null  datetime64[ns, UTC]
 1   temp_mean  43849 non-null  float32            
 2   temp_std   43849 non-null  float32            
dtypes: datetime64[ns, UTC](1), float32(2)
memory usage: 685.3 KB


ENGINEER FEATURES FOR LOAD

In [ ]:

# Engineer features
weather_load = add_lag_features_load(weather_load)
weather_load = add_rolling_features_load(weather_load)
weather_load = add_time_features(weather_load)
weather_load = weather_load.dropna()

print(f"  Weather features: {weather_load.shape}")
print(f"  Columns: {weather_load.columns.tolist()}")



  Weather features: (43825, 15)
  Columns: ['ds', 'temp_mean', 'temp_std', 'temp_lag1', 'temp_lag24', 'temp_roll_3h', 'temp_roll_6h', 'temp_roll_24h', 'temp_roll_3h_std', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']


CREATE LOAD DATAFRAME

In [ ]:

weather_load['ds'] = pd.to_datetime(weather_load['ds'])
weather_load = weather_load.set_index('ds')

# Combine target + weather + network
load_df = target_train[['FR_load_actual']].join(weather_load)
load_df = load_df.join(network_train)
load_df = load_df.dropna()

print(f"  Load dataframe shape: {load_df.shape}")
print(f"  Total features: {load_df.shape[1] - 1}")



  Load dataframe shape: (43761, 24)
  Total features: 23


SPLIT DATA

In [ ]:

X_train = load_df.loc['2020':'2023'].drop('FR_load_actual', axis=1)
y_train = load_df.loc['2020':'2023']['FR_load_actual']

X_valid = load_df.loc['2024'].drop('FR_load_actual', axis=1)
y_valid = load_df.loc['2024']['FR_load_actual']

In [ ]:

print(f"  X_train: {X_train.shape}")
print(f"  X_valid: {X_valid.shape}")

  X_train: (35011, 23)
  X_valid: (8749, 23)


## MODELS

XGBOOST

In [ ]:

model_load = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=42,
    tree_method='hist',
    verbosity=0
)

model_load.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=10, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:


y_pred_train = model_load.predict(X_train)
y_pred_valid = model_load.predict(X_valid)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
mae_train = mean_absolute_error(y_train, y_pred_train)
wmape_train = np.sum(np.abs(y_train - y_pred_train)) / np.sum(np.abs(y_train))
mape_train = mean_absolute_percentage_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)

rmse_valid = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
mae_valid = mean_absolute_error(y_valid, y_pred_valid)
mape_valid = mean_absolute_percentage_error(y_valid, y_pred_valid)
wmape_valid = np.sum(np.abs(y_valid - y_pred_valid)) / np.sum(np.abs(y_valid))
r2_valid = r2_score(y_valid, y_pred_valid)

print(f"\nTraining: RMSE = {rmse_train:.2f} MW ,MAE: {mae_train:.2f} MW , WMAPE: {wmape_train:.4f} ({wmape_train*100:.2f}%), MAPE: {mape_train:.4f} ({mape_train*100:.2f}%) R²: {r2_train:.4f}")
print(f"\nValidation:")
print(f"  RMSE: {rmse_valid:.2f} MW")
print(f"  MAE: {mae_valid:.2f} MW")
print(f"  WMAPE: {wmape_valid:.4f} ({wmape_valid*100:.2f}%)")
print(f"  MAPE : {mape_valid:.4f} ({mape_valid*100:.2f}%) ")
print(f"  R²: {r2_valid:.4f}")




Training: RMSE = 1646.44 MW ,MAE: 1230.62 MW , WMAPE: 0.0242 (2.42%), MAPE: 0.0248 (2.48%) R²: 0.9779

Validation:
  RMSE: 2586.73 MW
  MAE: 1937.86 MW
  WMAPE: 0.0396 (3.96%)
  MAPE : 0.0400 (4.00%) 
  R²: 0.9294


The model performs well on both training and validation sets, indicated by high R² values and relatively low error metrics. There is a slight drop in performance from training to validation, which is normal and suggests good generalization without significant overfitting.

CROSS-VALIDATION

In [ ]:


tscv = TimeSeriesSplit(n_splits=3)
X_cv = load_df.drop('FR_load_actual', axis=1)
y_cv = load_df['FR_load_actual']

rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)

rmse_scores = -cross_val_score(model_load, X_cv, y_cv, cv=tscv, scoring=rmse_scorer)
r2_scores = cross_val_score(model_load, X_cv, y_cv, cv=tscv, scoring='r2')

print(f"\nCross-Validation:")
print(f"  RMSE per fold: {rmse_scores}")
print(f"  Average RMSE: {rmse_scores.mean():.2f} MW")
print(f"  R² per fold: {r2_scores}")
print(f"  Average R²: {r2_scores.mean():.4f}")


Cross-Validation:
  RMSE per fold: [3067.9519227  3516.36858136 2601.95368906]
  Average RMSE: 3062.09 MW
  R² per fold: [0.92099023 0.87524903 0.92675745]
  Average R²: 0.9077


In [ ]:

print("Saving the model")


with open('load_final.pkl', 'wb') as f:
    pickle.dump(model_load, f)



Saving the model
